# Train actor-critic (M4 — frozen world model)

Imagination only: load the **700k** size-S world model, freeze it, train actor +
critic on 15-step `z_prior` rollouts. **No** `env.step`. That is M5.

Config: `configs/m4_actor_critic.yaml`. World-model weights:
`checkpoints/m3_dreamer_s/ckpt_step_700000.pt`. Replay is the same 600-episode
dump the world model trained on.

M4 bar: finite imagined rewards, actor-critic losses trend down, decoded
imagination looks like Crafter for a handful of steps then degrades. **Not** a
Crafter score. Sparse random-policy reward means the actor may not “get good
at the game”; entropy staying alive + critic loss falling is the pass.

`RESUME = "auto"` resumes the **actor-critic** checkpoint, not the world model.
Do not set the world-model path to `RESUME = None` and train a new RSSM here.


In [ ]:
from __future__ import annotations

import gc
import importlib
import json
import os
import random
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "src").is_dir():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            ROOT = candidate
            break
os.chdir(ROOT)
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
scripts = str(ROOT / "scripts")
if scripts not in sys.path:
    sys.path.insert(0, scripts)

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import clear_output, display
from torch.utils.tensorboard import SummaryWriter

import agents.actor_critic as ac_mod
import models.world_model as wm_mod
import training.ac_step as ac_step_mod
import training.device as device_mod
import training.imagine as imagine_mod
import training.replay_buffer as replay_mod
import training.returns as returns_mod
import training.ckpt as ckpt_mod

for mod in (
    ac_mod, wm_mod, ac_step_mod, device_mod, imagine_mod,
    replay_mod, returns_mod, ckpt_mod,
):
    importlib.reload(mod)

from agents.actor_critic import Actor, Critic
from training.ac_step import actor_critic_step
from training.ckpt import resolve_resume
from training.device import (
    configure_runtime,
    describe_device,
    get_device,
    make_grad_scaler,
    parse_amp,
    vram_peak_gb,
    warn_if_not_cuda,
)
from training.imagine import decode_imagination, freeze_world_model
from training.replay_buffer import ReplayBuffer
from training.returns import PercentileReturnNorm
from train_actor_critic import (
    load_frozen_world_model,
    save_imagination_gif,
    save_imagination_strip,
)

CONFIG = Path("configs/m4_actor_critic.yaml")
RESUME = "auto"
STEPS_OVERRIDE = None

with CONFIG.open() as f:
    cfg = yaml.safe_load(f)
train_cfg = cfg["train"]

random.seed(int(cfg["seed"]))
np.random.seed(int(cfg["seed"]))
torch.manual_seed(int(cfg["seed"]))

device = get_device()
configure_runtime(device)
warn_if_not_cuda(device)
if device.type == "cuda":
    torch.cuda.manual_seed_all(int(cfg["seed"]))
print("device:", describe_device(device))
print(
    f"amp={train_cfg.get('amp')} batch={train_cfg['batch_size']} "
    f"seq={train_cfg['seq_len']} horizon={train_cfg['horizon']} "
    f"start_mode={train_cfg.get('start_mode')} "
    f"actor_lr={train_cfg['actor_lr']} entropy={train_cfg['entropy_scale']}"
)


In [ ]:
world_model, wm_cfg = load_frozen_world_model(cfg, device)
replay_path = Path(wm_cfg["collect"]["out_path"])
if not replay_path.is_file():
    raise FileNotFoundError(f"missing replay {replay_path}")
buffer = ReplayBuffer(seed=int(cfg["seed"]))
buffer.load_state_dict(torch.load(replay_path, weights_only=False))
print(f"replay: episodes={len(buffer)} steps={buffer.num_steps}")

actor_cfg = cfg.get("actor", {})
critic_cfg = cfg.get("critic", {})
actor = Actor(
    world_model.feat_dim,
    world_model.rssm.action_dim,
    hidden=int(actor_cfg.get("hidden", 512)),
    layers=int(actor_cfg.get("layers", 2)),
    unimix=float(actor_cfg.get("unimix", 0.01)),
).to(device)
critic = Critic(
    world_model.feat_dim,
    hidden=int(critic_cfg.get("hidden", 512)),
    layers=int(critic_cfg.get("layers", 2)),
    num_bins=int(critic_cfg.get("num_bins", 255)),
    low=float(critic_cfg.get("low", -20.0)),
    high=float(critic_cfg.get("high", 20.0)),
).to(device)
freeze_world_model(world_model)

optim = torch.optim.Adam(
    [
        {"params": actor.parameters(), "lr": float(train_cfg["actor_lr"])},
        {"params": critic.parameters(), "lr": float(train_cfg["critic_lr"])},
    ]
)
retnorm = PercentileReturnNorm()
amp_dtype = parse_amp(train_cfg.get("amp", "bf16"), device)
scaler = make_grad_scaler(device, amp_dtype)

ckpt_dir = Path(train_cfg["checkpoint_dir"])
results_dir = Path(train_cfg["results_dir"])
log_dir = Path(train_cfg["log_dir"])
for p in (ckpt_dir, results_dir, log_dir):
    p.mkdir(parents=True, exist_ok=True)

start_step = 0
resume_path = resolve_resume(RESUME, ckpt_dir)
if resume_path is not None:
    ac_ckpt = torch.load(resume_path, weights_only=False, map_location=device)
    actor.load_state_dict(ac_ckpt["actor"])
    critic.load_state_dict(ac_ckpt["critic"])
    if "optim" in ac_ckpt:
        optim.load_state_dict(ac_ckpt["optim"])
    if "retnorm" in ac_ckpt:
        retnorm.load_state_dict(ac_ckpt["retnorm"])
    start_step = int(ac_ckpt.get("step", 0))
    print(f"resumed actor-critic from {resume_path} at step {start_step}")
else:
    print("fresh actor-critic (world model is the frozen 700k M3 checkpoint)")

n_wm = sum(p.numel() for p in world_model.parameters()) / 1e6
n_ac = sum(p.numel() for p in list(actor.parameters()) + list(critic.parameters())) / 1e6
print(f"params: world_model={n_wm:.2f}M (frozen)  actor+critic={n_ac:.2f}M")


In [ ]:
writer = SummaryWriter(log_dir=str(log_dir))
steps = int(STEPS_OVERRIDE) if STEPS_OVERRIDE is not None else int(train_cfg["steps"])
batch_size = int(train_cfg["batch_size"])
seq_len = int(train_cfg["seq_len"])
horizon = int(train_cfg["horizon"])
start_mode = str(train_cfg.get("start_mode", "all"))
log_every = int(train_cfg["log_every"])
image_every = int(train_cfg["image_every"])
ckpt_every = int(train_cfg["checkpoint_every"])

history: list[dict] = []
metrics_path = results_dir / "train_metrics.json"
if start_step > 0 and metrics_path.is_file():
    prev = json.loads(metrics_path.read_text(encoding="utf-8"))
    history = [h for h in prev if int(h.get("step", 0)) <= start_step]
    print(f"loaded {len(history)} logged points up to step {start_step}", flush=True)

last_vis = None
last_log_time = time.time()
last_log_step = start_step


def _thin_history(hist: list[dict], max_points: int = 800) -> list[dict]:
    if len(hist) <= max_points:
        return hist
    stride = max(1, len(hist) // max_points)
    return hist[::stride]


def _dashboard_strip(images: np.ndarray, keep_every: int = 2) -> np.ndarray:
    """images [H, 64, 64, 3] uint8 → thinner strip for imshow only."""
    tiles = [images[i] for i in range(0, images.shape[0], keep_every)]
    return np.concatenate(tiles, axis=1)


def show_progress(hist, vis_nchw, steps_per_sec=None):
    status = ""
    if hist:
        h = hist[-1]
        rate = f"  ({steps_per_sec:.2f} steps/s)" if steps_per_sec else ""
        status = (
            f"step {h['step']}/{steps}  total={h['total']:.4f}  "
            f"actor={h['actor']:.4f} critic={h['critic']:.4f}  "
            f"H={h['entropy']:.3f}  ret={h['return']:.4f}{rate}"
        )
    try:
        clear_output(wait=False)
        plot_h = _thin_history(hist)
        fig = plt.figure(figsize=(11, 6), dpi=80)
        ax0 = fig.add_subplot(2, 2, 1)
        xs = [p["step"] for p in plot_h]
        ax0.plot(xs, [p["critic"] for p in plot_h], label="critic", color="#1f4e79")
        ax0.plot(xs, [p["actor"] for p in plot_h], label="actor", color="#c45c26", alpha=0.8)
        ax0.set_title("actor / critic loss")
        ax0.legend(fontsize=8)
        ax1 = fig.add_subplot(2, 2, 2)
        ax1.plot(xs, [p["entropy"] for p in plot_h], color="#2a9d8f")
        ax1.set_title("policy entropy (nats)")
        ax2 = fig.add_subplot(2, 2, 3)
        ax2.plot(xs, [p["return"] for p in plot_h], color="#264653")
        ax2.set_title("imagined λ-return (mean)")
        if vis_nchw is not None:
            ax3 = fig.add_subplot(2, 2, 4)
            seq = vis_nchw[0].detach().float().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
            ax3.imshow(_dashboard_strip((seq * 255).astype(np.uint8)))
            ax3.set_title("imagined decode (thinned)", fontsize=9)
            ax3.axis("off")
        fig.tight_layout()
        display(fig)
    except Exception as exc:
        print(f"[dashboard skipped: {type(exc).__name__}: {exc}]", flush=True)
    finally:
        plt.close("all")
        gc.collect()
    if status:
        print(status, flush=True)


print(
    f"training actor-critic to step {steps} on {device} "
    f"(start={start_step}, remaining={steps - start_step})...",
    flush=True,
)
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for step in range(start_step + 1, steps + 1):
    batch = buffer.sample(batch_size, seq_len)
    _loss, metrics, rollout = actor_critic_step(
        world_model,
        actor,
        critic,
        optim,
        batch,
        device=device,
        retnorm=retnorm,
        horizon=horizon,
        start_mode=start_mode,
        lam=float(train_cfg.get("lam", 0.95)),
        discount=float(train_cfg.get("discount", 0.997)),
        entropy_scale=float(train_cfg.get("entropy_scale", 3.0e-4)),
        amp_dtype=amp_dtype,
        scaler=scaler,
        max_grad_norm=float(train_cfg.get("max_grad_norm", 100.0)),
    )
    metrics["step"] = step

    if step % log_every == 0 or step == start_step + 1:
        now = time.time()
        sps = (step - last_log_step) / max(now - last_log_time, 1e-6)
        last_log_time = now
        last_log_step = step
        metrics["steps_per_sec"] = sps
        history.append(metrics)
        for k, v in metrics.items():
            if k != "step":
                writer.add_scalar(f"ac/{k}", v, step)
        vram = vram_peak_gb()
        if vram:
            metrics["vram_alloc"] = vram[0]
        show_progress(history, last_vis, steps_per_sec=sps)

    if step % image_every == 0 or step == start_step + 1:
        last_vis = decode_imagination(world_model, rollout.feat, max_starts=1)
        save_imagination_strip(last_vis, results_dir / f"imagine_step_{step:06d}.png")
        save_imagination_gif(last_vis, results_dir / f"imagine_step_{step:06d}.gif")

    if step % ckpt_every == 0:
        payload = {
            "step": step,
            "actor": actor.state_dict(),
            "critic": critic.state_dict(),
            "optim": optim.state_dict(),
            "retnorm": retnorm.state_dict(),
        }
        torch.save(payload, ckpt_dir / f"ckpt_step_{step}.pt")
        torch.save(payload, ckpt_dir / "ckpt_latest.pt")
        metrics_path.write_text(json.dumps(history), encoding="utf-8")
        print(f"wrote {ckpt_dir / f'ckpt_step_{step}.pt'}", flush=True)

final = {
    "step": steps,
    "actor": actor.state_dict(),
    "critic": critic.state_dict(),
    "optim": optim.state_dict(),
    "retnorm": retnorm.state_dict(),
}
torch.save(final, ckpt_dir / "ckpt_final.pt")
torch.save(final, ckpt_dir / "ckpt_latest.pt")
metrics_path.write_text(json.dumps(history), encoding="utf-8")
last_vis = decode_imagination(world_model, rollout.feat, max_starts=1)
save_imagination_strip(last_vis, results_dir / "imagine_final.png")
save_imagination_gif(last_vis, results_dir / "imagine_final.gif")
writer.flush()
writer.close()
show_progress(history, last_vis)
print("done", ckpt_dir / "ckpt_final.pt")


## Exit criteria — M4, not a Crafter score

- Imagined rewards finite, not NaN, not exploding.
- Critic loss trends down; policy entropy stays above ~0.1 nats (not a delta policy).
- `imagine_final.gif` looks like Crafter for a handful of steps, then smears. Garbage from frame 1 is a fail (dynamics / freeze bug).

Look at `results/m4_actor_critic/imagine_final.png`. Do not wait for HUD digits or a positive Crafter return — that is M5 with online collect.


In [ ]:
assert len(history) >= 2, "need at least two logged points"

first, last = history[0], history[-1]
checks: list[tuple[str, bool, str]] = []

rew = float(last["reward"])
checks.append(
    (
        f"imagined reward is finite (last mean={rew:.4f})",
        np.isfinite(rew) and abs(rew) < 50.0,
        "imagined reward NaN or exploding — freeze/AMP/head bug",
    )
)
ent = float(last["entropy"])
checks.append(
    (
        f"policy entropy is alive (last={ent:.3f} > 0.1)",
        ent > 0.1,
        "actor collapsed to a delta — raise entropy_scale",
    )
)
c0, c1 = float(first["critic"]), float(last["critic"])
drop = 1.0 - (c1 / max(c0, 1e-8))
checks.append(
    (
        f"critic loss dropped or stayed bounded (first={c0:.3f} -> last={c1:.3f})",
        c1 < 20.0 and (drop > 0.0 or c1 < c0 + 1.0),
        "critic is diverging — check return normalization",
    )
)
vis = decode_imagination(world_model, rollout.feat, max_starts=1)
checks.append(
    (
        f"decoded imagination is not a constant frame (std={float(vis.std()):.4f})",
        float(vis.std()) > 0.02,
        "open-loop decode collapsed — inspect imagine_final.gif",
    )
)

print(f"steps trained: {last['step']}\n")
all_pass = True
for description, ok, hint in checks:
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {description}")
    if not ok:
        print(f"       -> {hint}")
        all_pass = False
print()
print("RESULT:", "PASS — M4 actor-critic looks healthy" if all_pass else "FAIL — see hints above")
print("GIF:", results_dir / "imagine_final.gif")
